# Ambient 测试 notebook

只做评测，不训练。给一个量化流式模型算误唤醒率（FAPH）、漏报率（FRR）和 ROC，并输出图表。

**两种数据来源，二选一**

1. **复用/重建训练 run 的 ambient 特征**：`features_v1` 里的 `negative/ambient`、`negative/ambient_chime6`。缺了就由第 4 节从同一份 prepared 音频重建，构建是确定性的、幂等的（逐条 SHA-256 校验，数据不变就不重算）。
2. **直接给音频目录**（不需要任何训练产物）：`AMBIENT_AUDIO = {"room": ["/path/to/wavs"], ...}`。特征用训练数据同一套 C 前端（10 ms 步长、40 bin、30 ms 窗）现场生成，因此不需要 TensorFlow，也不需要特征缓存。

**本 notebook 只需要**

- 一个量化流式 `.tflite`：自己的训练产物，或从 [micro-wake-word-models](https://github.com/esphome/micro-wake-word-models) 下载的模型。放 `trained_models/<run>/tflite_stream_state_internal_quant/`（第 4 节会从 Drive 复制），或用 `MICROWAKEWORD_MODEL` 指任意路径。
- 数据：默认从 Google Drive 的 `/content/drive/MyDrive/wake-word` 复制（第 4 节），再按需重建 ambient 特征（第 5 节，幂等）；也可以直接用任意音频目录。
- 本仓库，以及 `pymicro-features`、`ai-edge-litert`、`mmap_ninja`、`pandas`、`matplotlib`。
- **不 import TensorFlow**，也不需要训练 notebook 在本机跑过；它只是这些文件的产出者，不是运行前提。

**口径与训练一致**：10 ms 步长、首层 stride 3、5 帧滑动平均、误唤醒后 25 帧冷却、FAPH = 次数 ÷ 音频小时。数字可以和训练写出的 `tflite_streaming_roc.txt`、`chime6_ambient_results.json` 直接对照。只读输入：除了写 `ambient_eval/` 下的报告和图表，不改动任何数据。

## 1. 路径与评测参数

模型路径优先看 `MICROWAKEWORD_MODEL`；没设就在训练 run 目录里找，再找不到就搜 `run_root` 下任意量化流式模型。所以单独测试一个下载来的模型时，只要把 `MICROWAKEWORD_MODEL` 指向那个 `.tflite`。

In [1]:
from pathlib import Path
import os

repo_path = Path(os.environ.get("MICROWAKEWORD_REPO", "/content/micro-wake-word")).expanduser().resolve()
# 训练 run 目录：有 features_v1 就复用它；只用音频目录评测时可以不存在。
run_root = Path(os.environ.get("MICROWAKEWORD_RUN", repo_path)).expanduser().resolve()

wake_word_name = "hi_fondcircle"
tts_version = "clean100_v1"
positive_dataset_name = f"{tts_version}_three_speeds"
run_suffix = f"_{tts_version}_hey_hi"

data_dir = run_root / "data_v1"          # licensed_data 的 downloads/ 与 prepared/ 在这里
features_dir = run_root / "features_v1"
positive_dir = features_dir / f"positive_{wake_word_name}_{positive_dataset_name}"
# 两个池子都存在 <pool>/<mode>/*_mmap 下：FSD50K 噪声 + LibriSpeech 语音，以及 CHiME-6。
ambient_feature_dirs = [
    features_dir / "negative" / "ambient",
    features_dir / "negative" / "ambient_chime6",
]
train_dir = run_root / "trained_models"
positives_mmap = positive_dir / "testing" / "wakeword_mmap"

# 模型放哪都行，按这个顺序找：
#   1. MICROWAKEWORD_MODEL 指定的 .tflite（也可以是 Drive 里的路径）
#   2. run 目录里的训练产物：trained_models/<run>/tflite_stream_state_internal_quant/
#   3. run 目录下任意 *stream*quant*.tflite
requested_model = os.environ.get("MICROWAKEWORD_MODEL")
default_model = train_dir / "hi_circle_v1.tflite"
model_candidates = [Path(requested_model).expanduser() if requested_model else default_model]
if run_root.is_dir():
    model_candidates += sorted(run_root.rglob("*stream*quant*.tflite"))
model_path = next((path for path in model_candidates if path.is_file()), model_candidates[0])

# 推理参数必须和训练/量化时一致，否则数字不能与 tflite_streaming_roc.txt 对比。
stride = 3                 # mixednet 首层卷积步长
step_ms = 10               # 特征步长
sliding_window_length = 5  # 滑动平均帧数

AMBIENT_MODE = "testing_ambient"  # 想看 dev 侧（4.46 h）就换成 "validation_ambient"
MAX_TRACKS = None                 # 冒烟测试时可设 5，只读每个源的前几条音频
REPORT_CUTOFFS = (0.90, 0.95, 0.98, 0.99)
eval_dir = train_dir / "ambient_eval"

print(f"repo_path = {repo_path}")
print(f"run_root  = {run_root}")
print(f"model     = {model_path}")
print(f"positives = {positives_mmap}")

repo_path = /content/micro-wake-word
run_root  = /content/micro-wake-word
model     = /content/micro-wake-word/trained_models/hi_circle_v1.tflite
positives = /content/micro-wake-word/features_v1/positive_hi_fondcircle_clean100_v1_three_speeds/testing/wakeword_mmap


## 2. 环境

只在缺东西时动手：仓库不在就 clone，包不能 import 就 `pip install -e`，缺 matplotlib 才装。

In [2]:
import importlib.util
import subprocess
import sys

if not repo_path.is_dir():
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/Seeker32/micro-wake-word.git", str(repo_path)],
                   check=True)
os.chdir(repo_path)

if importlib.util.find_spec("microwakeword") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_path)], check=True)
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib"], check=True)
print("环境就绪")

环境就绪


## 3. 选数据来源

默认走「复用训练特征」：`features_v1` 存在就直接用，不额外花钱。想评测别的音频（自己的房间录音、下载的语料、某个 TTS 生成的正样本）时，把 `AMBIENT_AUDIO` / `POSITIVE_AUDIO` 填上音频目录，就走现场算特征那条路。

- `AMBIENT_AUDIO`：`{源名: [目录或文件, ...]}`，每个源单独出 FAPH，再汇总。目录会递归找 `.wav/.flac/.ogg`。
- `POSITIVE_AUDIO`：唤醒词音频（真人录音或 TTS 都行）。原始录音里词在中间，所以**不跳过开头**；训练生成的正样本是左填充的，仍然跳过 25 帧。

In [3]:
from microwakeword.audio.ambient_eval import (
    AudioTracks, find_ambient_sets, find_audio_files,
)

# 填了就优先用音频，忽略 features_v1。示例：
# AMBIENT_AUDIO = {"living_room": ["/content/my_room_recordings"],
#                  "tv": ["/content/tv_audio"]}
AMBIENT_AUDIO = None
# 例如自己录的 20 条唤醒词：POSITIVE_AUDIO = ["/content/wake_word_records"]
POSITIVE_AUDIO = None


def resolve_ambient_sources():
    if AMBIENT_AUDIO is not None:
        return {label: AudioTracks(find_audio_files(*paths))
                for label, paths in AMBIENT_AUDIO.items()}
    return find_ambient_sets(*ambient_feature_dirs, mode=AMBIENT_MODE)


def resolve_positives():
    if POSITIVE_AUDIO is not None:
        return AudioTracks(find_audio_files(*POSITIVE_AUDIO)), 0
    return positives_mmap, 25  # 训练生成的正样本左侧有填充帧

# 真正的解析放在第 5 节：先让第 4 节把缺的 ambient 特征建出来，再列清单。

ModuleNotFoundError: No module named 'microwakeword.audio.ambient_eval'

## 4. 数据集：默认从 Google Drive 复制

Colab 每次都是空机器，重新下载约 11 GiB 压缩包（解压后约 15 GiB 音频）太慢，所以默认从 Drive 里已有的副本复制。复制本身是幂等的：大小一致的文件直接跳过，字节先写 `.part` 再改名（中断不会留下半截压缩包），FSD50K / LibriSpeech 的官方 MD5 会在复制后核对，坏的副本直接删掉并报错。

`prepared/` 的深度校验仍由 `prepare_licensed_data` / `ensure_chime6` 负责——它们加载时会按 manifest 逐文件比 SHA-256。

Drive 目录下认这些条目（有哪个复制哪个，都没有就跳过）：

| Drive 里的路径 | 复制到 | 说明 |
| --- | --- | --- |
| `trained_models` | `trained_models` | 训练产物，含量化流式 `.tflite` |
| `data_v1/prepared` | `data_v1/prepared` | 已挑选好的音频，最省事 |
| `data_v1/downloads` | `data_v1/downloads` | 8 个原始压缩包，本机解压 |
| `chime6_v1/prepared` | `chime6_v1/prepared` | 已解压的 CHiME-6 |
| `chime6_v1.zip` | `data_v1/downloads/chime6_v1.zip` | `ensure_chime6` 认得的位置 |
| `features_v1/negative/ambient*` | 同路径 | 已建好的 ambient 特征；没有 prepared 音频时第 5 节会跳过构建、直接复用 |

In [4]:
DATA_FROM_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/wake-word")  # 放 data_v1/、chime6_v1.zip、features_v1/ 的目录
VERIFY_COPIED_ARCHIVES = True

import pandas as pd

from microwakeword.audio.drive_data import archives_present, mount_drive, sync_datasets

if DATA_FROM_DRIVE and mount_drive() and DRIVE_ROOT.is_dir():
    report = sync_datasets(DRIVE_ROOT, run_root, verify=VERIFY_COPIED_ARCHIVES)
    rows = [{"entry": entry, "present": stats["present"],
             "files": stats.get("files", 0), "copied": stats.get("copied", 0),
             "skipped": stats.get("skipped", 0),
             "GiB": round(stats.get("bytes", 0) / 1024 ** 3, 2)}
            for entry, stats in report.items()]
    display(pd.DataFrame(rows).set_index("entry"))
else:
    print(f"跳过 Drive 复制（DATA_FROM_DRIVE={DATA_FROM_DRIVE}，Drive={DRIVE_ROOT}）")

# 只有压缩包时在本机解压出 prepared/。已有 prepared 就跳过——它是否过期由
# 第 5 节用 manifest 摘要判断，不必每次重跑解压。
if archives_present(data_dir / "downloads") and not (data_dir / "prepared").is_dir():
    from microwakeword.audio.licensed_data import prepare_licensed_data

    print(f"解压 {data_dir / 'downloads'} ...")
    prepare_licensed_data(str(data_dir))

ModuleNotFoundError: No module named 'microwakeword.audio.drive_data'

## 5. 只生成 ambient 特征（幂等）

`microwakeword.audio.ambient_features` 只建 ambient 评测集，从不碰训练特征：

- **确定性**：按 path 排序逐条处理，无增强、无打乱、无重复、无随机切块；同一份数据集 + 同一组参数，重建出来的 track 逐字节相同。
- **幂等**：manifest 记录参数签名、prepared 数据集 manifest 的 SHA-256 和每条 track 的 SHA-256。再跑一次会先校验，命中就原样复用（不重算）；数据、参数或磁盘内容变了才重建，不会拿过期特征出报告。
- **原子**：先写临时目录再整体换入，中断不会留下半个集合。

源数据的完整性仍由 `prepare_licensed_data` / `ensure_chime6` 负责，这里只记录所依据的 manifest 摘要。缺 `data_v1/prepared` 时会跳过对应的语料并提示，而不是悄悄少算一部分。

In [ ]:
BUILD_AMBIENT_FEATURES = True   # False：直接用 features_v1 里已有的 ambient 特征
VERIFY_FEATURE_HASHES = True    # 逐条校验 track 哈希；信任签名可设 False，省几秒
ALLOW_CHIME6_DOWNLOAD = False   # 本地没有 chime6_v1 时是否去 OpenSLR 流式下载（约 12 GB）
CORPORA_TO_BUILD = ("noise", "speech", "chime6")
BUILD_SPLITS = (("validation", "validation_ambient"), ("testing", "testing_ambient"))

if AMBIENT_AUDIO is not None:
    print("用音频目录评测，跳过特征生成")
elif BUILD_AMBIENT_FEATURES:
    from microwakeword.audio.ambient_features import build_all

    from microwakeword.audio.chime6 import ensure_chime6, find_chime6_source

    licensed_prepared = data_dir / "prepared"
    chime6_dir = run_root / "chime6_v1"
    chime6_prepared = chime6_dir / "prepared"
    local_chime6 = find_chime6_source(data_dir / "downloads", run_root)
    if not chime6_prepared.is_dir() and (local_chime6 is not None or ALLOW_CHIME6_DOWNLOAD):
        chime6_prepared = ensure_chime6(chime6_dir,
                                       source_dirs=[data_dir / "downloads", run_root])

    roots = {"noise": licensed_prepared, "speech": licensed_prepared,
             "chime6": chime6_prepared}
    corpora = tuple(corpus for corpus in CORPORA_TO_BUILD if roots[corpus].is_dir())
    skipped = sorted(set(CORPORA_TO_BUILD) - set(corpora))
    if skipped:
        print(f"跳过（没有 prepared 音频）：{skipped}；"
              "先准备好数据（第 4 节的 Drive 复制），或设置 ALLOW_CHIME6_DOWNLOAD=True")
    if corpora:
        manifests = build_all(
            features_dir,
            licensed_prepared=licensed_prepared if corpora != ("chime6",) else None,
            chime6_prepared=chime6_prepared if "chime6" in corpora else None,
            corpora=corpora,
            splits=BUILD_SPLITS,
            verify=VERIFY_FEATURE_HASHES,
        )
        for pool, manifest in manifests.items():
            hours = ", ".join(f"{mode}={value:.2f}h"
                              for mode, value in manifest["hours"].items())
            print(f"{pool}: {hours}")

## 6. 检查清单

缺文件时这里直接报错，说明缺什么、回哪里补，而不是跑到一半才失败。清单里的时长是特征帧数（或音频头信息）换算的真实音频时长，`tracks` 是条数。

In [ ]:
import json

import pandas as pd

from microwakeword.audio.ambient_eval import summarize_source

sources = resolve_ambient_sources()
positives, positives_skip_first = resolve_positives()
print(f"ambient 源：{list(sources)}")

if not model_path.is_file():
    tried = "\n  ".join(str(path) for path in dict.fromkeys(model_candidates))
    raise FileNotFoundError(
        "找不到量化流式模型。三种放法任选其一：\n"
        "  1. 环境变量 MICROWAKEWORD_MODEL=/path/to/stream_state_internal_quant.tflite\n"
        "  2. 放进 run 目录的 trained_models/<run>/tflite_stream_state_internal_quant/"
        "（第 4 节会连 trained_models 一起从 Drive 复制）\n"
        "  3. 直接指向 Drive 里的路径（已挂载）\n"
        f"已尝试：\n  {tried}"
    )

rows = []
for label, source in sources.items():
    summary = summarize_source(source)
    rows.append({"source": label, "tracks": summary["tracks"],
                 "hours": round(summary["hours"], 2),
                 "input": "; ".join(summary["inputs"])})
inventory = pd.DataFrame(rows).set_index("source")
display(inventory)

positive_summary = summarize_source(positives)
print(f"{AMBIENT_MODE} 合计 {inventory['hours'].sum():.2f} h / {inventory['tracks'].sum()} 条；"
      f"正样本 {positive_summary['tracks']} 条")

## 7. 跑评测

每条音频都用**全新的解释器实例**推理，避免流式状态从一段录音带到下一段（会把误唤醒算多或算少）。所以这里的数字和 `evaluate_ambient_model`（CHiME-6 单独报告）完全同口径，可以直接对照。

CHiME-6 的 `testing_ambient` 有 15.6 h，大约要几分钟到十几分钟；想先看流程能不能跑通，把上面的 `MAX_TRACKS` 设成 5。

In [ ]:
import time

from microwakeword.audio.ambient_eval import evaluate_ambient

started = time.monotonic()
evaluated = {"tracks": 0}


def progress(label, done, total):
    evaluated["tracks"] += 1
    if evaluated["tracks"] % 5 == 0:
        print(f"  {label}: 第 {done} 条，累计 {evaluated['tracks']} 条，"
              f"{time.monotonic() - started:.0f}s", flush=True)


report = evaluate_ambient(
    model_path,
    sources,
    stride=stride,
    step_ms=step_ms,
    window=sliding_window_length,
    max_tracks=MAX_TRACKS,
    positives=positives,
    positives_skip_first=positives_skip_first,
    max_faph=2.0,
    progress=progress,
)
report["mode"] = AMBIENT_MODE
report["feature_dirs"] = [str(path) for path in ambient_feature_dirs]
print(f"完成：ambient {report['pooled']['hours']:.2f} h / {evaluated['tracks']} 条，"
      f"正样本 {report['positives']['count']} 条，用时 {time.monotonic() - started:.0f}s")
if report.get("roc"):
    print(f"pooled AUC = {report['roc']['auc']:.4f}（0–2 FAPH 区间内的曲线下面积，"
          "和训练日志同定义；越小越好，不是归一化的 ROC AUC）")

## 8. 怎么看这些数字

- **FAPH**：每小时误唤醒次数。只在有真实房间录音时才有意义；FSD50K 噪声片段偏干净、LibriSpeech 是近讲朗读，两者都会低估家庭场景，所以 CHiME-6（或你自己录的房间）那一列最值得看。
- **FRR**：唤醒词没被触发（峰值概率没过阈值）的比例。用 TTS 生成的正样本会偏乐观，真人口音/语速明显更差。
- **FAPH 与 FRR 要一起看**：阈值提高，两者都降；只看一边总能挑出好看的数。
- **per-source FAPH 相加 ≠ pooled FAPH**：pooled 是「总次数 ÷ 总小时」，各源时长不同不能直接相加；表里同时给出两者，便于定位问题来源。
- 某条音频整条峰值都很高，通常意味着出现了接近的发音或持续的电视/音乐，可以到后面的「最响片段」表里定位。

In [ ]:
source_labels = list(report["sources"])
# np.arange 出来的阈值是 0.9500000000000001 这种，索引一律按两位小数对齐。
cutoff_index = {round(float(cutoff), 2): index for index, cutoff in enumerate(report["cutoffs"])}
rows = [row for row in report["summary"] if round(row["cutoff"], 2) in REPORT_CUTOFFS]
summary = pd.DataFrame(rows).set_index("cutoff")
summary = summary[["false_negative_rate", "false_accepts_per_hour"]
                  + [f"{label}_faph" for label in source_labels]]
summary.columns = ["FRR", "FAPH(pooled)"] + [f"FAPH({label})" for label in source_labels]
display(summary.round(4))

trigger_rows = []
for label, source in report["sources"].items():
    for cutoff in REPORT_CUTOFFS:
        index = cutoff_index[round(cutoff, 2)]
        trigger_rows.append({
            "source": label,
            "cutoff": cutoff,
            "hours": round(source["hours"], 2),
            "tracks_with_any_trigger": source["tracks_with_any_trigger"][index],
            "tracks": source["track_count"],
        })
triggers = pd.DataFrame(trigger_rows).pivot(index="cutoff", columns="source")["tracks_with_any_trigger"]
display(triggers)
print(f"正样本 {report['positives']['count']} 条；ambient 合计 {report['pooled']['hours']:.2f} h。"
      "触发条数按音频条数计，一条音频多次触发只算一次。")

## 9. ROC：误唤醒率 vs 漏报率

实线是全部 ambient 汇总，虚线是单源。同一个 FRR 下 FAPH 越低越好；曲线越靠左下越适合上设备。标记点是 `REPORT_CUTOFFS` 里各阈值的位置。

这里的 AUC 是 0–2 FAPH 区间内曲线下面积（训练日志里的同一个定义），**越小越好**，不是归一化到 0–1 的 ROC AUC：完美模型 AUC = 0，模型没用时 AUC 会趋近 2。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from microwakeword.audio.ambient_eval import roc_curve

figures = {}

roc = report.get("roc")
if roc is None:
    raise RuntimeError("没有正样本概率就算不出 ROC")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(roc["faph"], roc["frr"], marker="o", ms=3, lw=1.4,
        label=f"all ambient, pooled (AUC={roc['auc']:.4f})")
# 一次误唤醒都没有时各源曲线都挤在左边缘，画出来只会互相遮挡。
no_accepts = not any(report["pooled"]["false_accepts_per_hour"])
for label, source in ({} if no_accepts else report["sources"]).items():
    try:
        curve = roc_curve(source["false_accepts_per_hour"],
                          report["positives"]["false_negative_rate"],
                          report["cutoffs"], max_faph=2.0)
    except (IndexError, ZeroDivisionError):
        print(f"跳过 {label}：在 0–2 FAPH 区间内没有可用工作点")
        continue
    ax.plot(curve["faph"], curve["frr"], lw=1, ls="--", label=f"{label} (AUC={curve['auc']:.4f})")

for cutoff in REPORT_CUTOFFS:
    for index, point in enumerate(roc["cutoffs"]):
        if abs(point - cutoff) < 1e-6:
            ax.plot([roc["faph"][index]], [roc["frr"][index]], "k*", ms=7)
            ax.annotate(f"{cutoff:.2f}", (roc["faph"][index], roc["frr"][index]),
                        textcoords="offset points", xytext=(4, -11), fontsize=8)
            break

# 图内文字用英文：默认 matplotlib 字体没有中文字形，中文标签在多数环境下会渲染成方块。
if no_accepts:
    ax.text(0.5, 0.42,
            f"no false accepts in {report['pooled']['hours']:.2f} h of ambient audio: "
            "the whole curve sits on the left edge",
            transform=ax.transAxes, ha="center", fontsize=9, color="0.35")

ax.set(xlabel="False accepts per hour (FAPH)", ylabel="False negative rate (FRR)",
       xlim=(-0.05, 2), ylim=(0, 1.02),
       title="ROC: ambient false accepts vs wake-word misses")
ax.grid(alpha=0.3)
ax.legend(loc="upper right", fontsize=9)
figures["roc"] = fig
plt.show()

## 10. 阈值扫描：FAPH 随阈值变化

纵轴对数，0 次/小时画在 1e-3 的位置只作为可视下限。常见目标是「真实房间录音上 ≤ 1 次/小时」，即曲线穿过 1 的那一列。

In [ ]:
cutoffs = np.array(report["cutoffs"])
floor = 1e-3

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(cutoffs, np.maximum(report["pooled"]["false_accepts_per_hour"], floor),
            lw=2.2, color="black", label="all ambient, pooled")
shades = plt.cm.tab10.colors
for index, (label, source) in enumerate(report["sources"].items()):
    ax.semilogy(cutoffs, np.maximum(source["false_accepts_per_hour"], floor),
                lw=1.1, color=shades[index % len(shades)],
                label=f"{label} ({source['hours']:.1f} h)")

for cutoff in REPORT_CUTOFFS:
    ax.axvline(cutoff, color="0.75", lw=0.8, ls=":")
ax.axhline(1.0, color="0.4", lw=0.9, ls="--")
ax.text(0.51, 1.15, "1 accept/hour", fontsize=8, color="0.35")

ax.set(xlabel="Detection threshold", ylabel="False accepts per hour (log)",
       title="FAPH vs threshold, per source and pooled")
# 阈值低于 0.5 时模型几乎一直在触发（图上直奔上方），横轴从 0.5 开始才看得清工作区。
ax.set_xlim(0.5, 1.0)
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=9)
figures["faph_vs_cutoff"] = fig
plt.show()

## 11. 峰值概率分布：正样本 vs ambient

每条音频只取一个峰值（5 帧滑动平均的最大值）。理想情况下正样本集中在 1 附近、ambient 集中在 0 附近，中间重叠区就是可用的阈值区间；重叠越少，能选的阈值越宽。

In [ ]:
bins = np.linspace(0, 1, 101)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(report["positives"]["peaks"], bins=bins, density=True, alpha=0.45,
        color="tab:green", label=f"positives (n={report['positives']['count']})")
for index, (label, source) in enumerate(report["sources"].items()):
    ax.hist(source["peaks"], bins=bins, density=True, alpha=0.45,
            color=shades[index % len(shades)],
            label=f"{label} (n={source['track_count']})")

ax.set(xlabel="Per-clip peak probability", ylabel="Density (log)", yscale="log",
       title="Peak probability distribution")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=9)
figures["peak_histogram"] = fig
plt.show()

for label, source in report["sources"].items():
    peaks = np.array(source["peaks"])
    print(f"{label:>8}: median {np.median(peaks):.4f}  p95 {np.percentile(peaks, 95):.4f}  "
          f"max {peaks.max():.4f}  share >0.5 {np.mean(peaks > 0.5):.2%}")

## 12. 最响的音频

按峰值概率排序，方便判断误唤醒来自噪音、语音还是家庭对话。复用训练特征时 CHiME-6 会带出 session / 房间 / 阵列，配合 `chime6_v1/prepared/manifest.csv` 可以定位到具体时间点；用音频目录时显示文件名。

In [ ]:
TOP_CLIPS = 5

# CHiME-6 的 lineage 与 mmap 条数一一对应（按 path 排序，同一 split 内顺序一致）。
chime6_metadata = {}
lineage_path = features_dir / "negative" / "ambient_chime6" / "feature_sources.json"
if AMBIENT_AUDIO is None and lineage_path.is_file():
    lineage = json.loads(lineage_path.read_text())
    split = AMBIENT_MODE.replace("_ambient", "")
    clips = [clip for clip in lineage.get("clips", []) if clip["split"] == split]
    expected = report["sources"].get("chime6", {}).get("track_count")
    if expected is not None and len(clips) == expected:
        chime6_metadata["chime6"] = clips
    elif expected is not None:
        print(f"注意：CHiME-6 lineage 有 {len(clips)} 条、mmap 有 {expected} 条，跳过元信息")

audio_paths = {}
for label, source in (sources.items() if AMBIENT_AUDIO is not None else {}):
    paths = [path for item in (source if isinstance(source, list) else [source])
             if hasattr(item, "paths") for path in item.paths]
    audio_paths[label] = paths

rows = []
for label, source in report["sources"].items():
    peaks = np.array(source["peaks"])
    for index in np.argsort(peaks)[::-1][:TOP_CLIPS]:
        row = {"source": label, "peak": round(float(peaks[index]), 4), "track": int(index)}
        clips = chime6_metadata.get(label)
        if clips is not None and index < len(clips):
            row.update({key: clips[index][key] for key in ("session", "room", "array", "start_s")})
        if label in audio_paths and index < len(audio_paths[label]):
            row["file"] = str(audio_paths[label][index])
        rows.append(row)

loudest = pd.DataFrame(rows)
display(loudest)

## 13. 保存报告

报告和图表都写进 `train_dir/ambient_eval/`（用音频目录评测时落在 `run_root/trained_models/...` 下），随训练目录一起归档，便于和 `tflite_streaming_roc.txt`、`chime6_ambient_results.json`、`hey_hi_test_results.json` 放在一处对比。

In [ ]:
eval_dir.mkdir(parents=True, exist_ok=True)
report_path = eval_dir / f"{AMBIENT_MODE}_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
summary.to_csv(eval_dir / f"{AMBIENT_MODE}_cutoffs.csv")
loudest.to_csv(eval_dir / f"{AMBIENT_MODE}_loudest_clips.csv", index=False)
for name, fig in figures.items():
    fig.savefig(eval_dir / f"{AMBIENT_MODE}_{name}.png", dpi=150, bbox_inches="tight")

print(f"报告：{report_path}")
print("\n".join(f"图表：{eval_dir / f'{AMBIENT_MODE}_{name}.png'}" for name in figures))

## 已知局限

- **阈值只是估算**：峰值概率来自公开 ambient 集和（TTS 或真人）正样本，真机麦克风、增益、回声都会让分布偏移；上线前仍要在目标房间录一段真实音频复核。
- **合成正样本偏乐观**：同一套 TTS 权重的三档语速覆盖面有限，真人口音/语速的 FRR 会更高。
- **CHiME-6 是家庭晚餐对话**，不含整段电视播放、无声道等典型误唤醒场景，FAPH 不能当作「家庭场景天花板」。
- **冷却窗口**（25 帧 ≈ 0.75 s 音频）会把密集误唤醒合并计数，与训练期 ROC 保持一致，但会略微低估极端嘈杂场景。
- **`MAX_TRACKS` 会缩小样本量**：限制条数后小时数变少，同样次数会算出更高的 FAPH，只适合冒烟测试，不能写进结论。
- **现场算特征**与训练用的是同一套 C 前端，但仍受音频重采样质量影响（非 16 kHz 输入会经 `scipy.signal.resample_poly` 转换）。